# 대화에서 관계 triple 추출하기

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `ConversationKGMemory` 대신, 현재는 관계를 애플리케이션 스키마로 명시하고
구조화 출력으로 `(subject, predicate, object)` triple을 추출한 뒤 Store나 전용
그래프 데이터베이스에 저장하는 구성이 더 투명합니다.

이 노트북은 LangGraph Store에 triple을 저장하고, 특정 엔티티와 연결된 edge를
결정적으로 조회한 뒤 모델의 답변 근거로 사용합니다.


In [1]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")
model = init_chat_model(MODEL_ID)


## 관계 스키마와 추출기

모델에는 문장에 명시된 관계만 추출하도록 지시합니다. 지식 그래프에 추론 결과까지
섞으면 사실과 추정의 경계가 흐려지기 때문입니다.


In [3]:
from pydantic import BaseModel, Field


class Triple(BaseModel):
    subject: str = Field(description="관계의 출발 엔티티")
    predicate: str = Field(description="간결한 관계 이름")
    object: str = Field(description="관계의 도착 엔티티 또는 값")


class KnowledgeTriples(BaseModel):
    triples: list[Triple]


triple_extractor = model.with_structured_output(KnowledgeTriples)

source_text = (
    "김셜리씨는 판교에 거주합니다. "
    "김셜리씨는 우리 회사의 신입 디자이너입니다. "
    "테디는 김셜리씨와 같은 회사에서 일하는 동료입니다."
)

knowledge = triple_extractor.invoke(
    [
        {
            "role": "system",
            "content": (
                "텍스트에 명시된 사실만 지식 그래프 triple로 추출하세요. "
                "각 관계는 subject, predicate, object로 표현하고 추측하지 마세요."
            ),
        },
        {"role": "user", "content": source_text},
    ]
)
knowledge


KnowledgeTriples(triples=[Triple(subject='김셜리씨', predicate='거주한다', object='판교'), Triple(subject='김셜리씨', predicate='직업', object='신입 디자이너'), Triple(subject='테디', predicate='동료이다', object='김셜리씨')])

## triple 저장 및 조회

내용 기반 key를 쓰면 같은 triple을 다시 저장해도 중복 문서가 생기지 않습니다.


In [4]:
from hashlib import sha256

from langgraph.store.memory import InMemoryStore

triple_store = InMemoryStore()
namespace = ("knowledge", "company-demo")


def triple_key(triple: Triple) -> str:
    canonical = f"{triple.subject}|{triple.predicate}|{triple.object}"
    return sha256(canonical.encode("utf-8")).hexdigest()[:16]


for triple in knowledge.triples:
    triple_store.put(namespace, triple_key(triple), triple.model_dump())


def facts_about(entity: str) -> list[dict]:
    items = triple_store.search(namespace, limit=100)
    return [
        item.value
        for item in items
        if entity in item.value["subject"] or entity in item.value["object"]
    ]


In [5]:
shirley_facts = facts_about("김셜리")
shirley_facts


[{'subject': '김셜리씨', 'predicate': '거주한다', 'object': '판교'},
 {'subject': '김셜리씨', 'predicate': '직업', 'object': '신입 디자이너'},
 {'subject': '테디', 'predicate': '동료이다', 'object': '김셜리씨'}]

## 조회한 edge만 근거로 답변하기


In [6]:
context = "\n".join(
    f"- {fact['subject']} --{fact['predicate']}--> {fact['object']}"
    for fact in shirley_facts
)

response = model.invoke(
    [
        {
            "role": "system",
            "content": (
                "아래 Relevant facts에 포함된 사실만 사용하세요. "
                "근거가 없으면 모른다고 답하세요.\n\n"
                f"Relevant facts:\n{context}"
            ),
        },
        {"role": "user", "content": "김셜리씨는 누구이며 어디에 거주합니까?"},
    ]
)
print(response.content)


김셜리씨는 **신입 디자이너**이며, **판교에 거주**합니다.


이 방식은 학습용 소규모 그래프에 적합합니다. 다중 hop 탐색, 역방향 index, 대규모
관계 질의가 핵심이라면 Neo4j 같은 전용 그래프 데이터베이스를 사용하고, LangGraph
Store에는 사용자 메모리나 조회 결과의 식별자를 보관하는 편이 좋습니다.
